In [ ]:
import base58
import hashlib
import os

# Example 1: Basic encoding and decoding operations
print("base58 package imported successfully")

# Encode text data
original_text = "Hello, Base58 encoding!"
text_bytes = original_text.encode('utf-8')
encoded = base58.b58encode(text_bytes)

print(f"Original text: {original_text}")
print(f"Encoded (Base58): {encoded.decode()}")

# Decode back to original
decoded_bytes = base58.b58decode(encoded)
decoded_text = decoded_bytes.decode('utf-8')

print(f"Decoded text: {decoded_text}")
assert decoded_bytes == text_bytes and decoded_text == original_text, decoded_text
print("Round-trip successful")
print("Example 1: Basic encoding and decoding demonstrated")

In [ ]:
import base58
import hashlib

# Example 2: Base58Check, the address format with a version byte and a checksum
# b58encode_check appends the first 4 bytes of SHA256(SHA256(data)) and encodes the result.
payload = hashlib.sha256(b"example public key").digest()[:20]  # a 20-byte stand-in for a key hash
versioned_payload = b"\x00" + payload  # version byte 0x00

address = base58.b58encode_check(versioned_payload).decode()
checksum = hashlib.sha256(hashlib.sha256(versioned_payload).digest()).digest()[:4]
assert address == base58.b58encode(versioned_payload + checksum).decode(), address
assert address.startswith("1"), address  # each leading zero byte encodes as "1"
assert base58.b58decode_check(address) == versioned_payload
print(f"Base58Check address: {address}")

# Changing one character breaks the checksum, and decoding refuses the address.
tampered = address[:-1] + ("2" if address[-1] != "2" else "3")
try:
    base58.b58decode_check(tampered)
except ValueError as e:
    print(f"Tampered address rejected: {e}")
else:
    raise AssertionError(f"b58decode_check accepted the tampered address {tampered}")

In [ ]:
import base58
import hashlib
import struct

# Example 3: Data integrity verification with checksums
def encode_with_checksum(data):
    """Encode data with checksum for integrity verification"""
    # Calculate checksum (first 4 bytes of SHA256)
    checksum = hashlib.sha256(data).digest()[:4]
    
    # Combine data with checksum
    data_with_checksum = data + checksum
    
    # Encode with Base58
    encoded = base58.b58encode(data_with_checksum)
    return encoded

def decode_with_verification(encoded_data):
    """Decode and verify data integrity"""
    try:
        # Decode from Base58
        decoded = base58.b58decode(encoded_data)
        
        # Split data and checksum
        data = decoded[:-4]
        checksum = decoded[-4:]
        
        # Verify checksum
        expected_checksum = hashlib.sha256(data).digest()[:4]
        
        if checksum == expected_checksum:
            return data, True
        else:
            return None, False
    except Exception:
        return None, False

# Test data integrity
test_data = b"Important financial transaction data"
encoded = encode_with_checksum(test_data)
decoded, valid = decode_with_verification(encoded)

print(f"Original data: {test_data}")
print(f"Encoded: {encoded.decode()}")
print(f"Decoded: {decoded}")
assert valid and decoded == test_data, (valid, decoded)
print("Integrity check: PASS")

# Flipping one bit of the payload makes the stored checksum disagree
corrupted = bytearray(base58.b58decode(encoded))
corrupted[0] ^= 1
assert decode_with_verification(base58.b58encode(bytes(corrupted))) == (None, False)
print("Corrupted payload rejected")

In [ ]:
import base58
import time

# Example 4: Bulk data processing and performance
def process_data_batch(data_items):
    """Process multiple items with Base58 encoding"""
    results = []
    start_time = time.time()
    
    for i, item in enumerate(data_items):
        # Simulate data processing
        processed_data = f"item_{i:04d}_{item}".encode('utf-8')
        encoded = base58.b58encode(processed_data)
        
        # Store result with metadata
        results.append({
            'id': i,
            'original': item,
            'encoded': encoded.decode(),
            'size': len(encoded)
        })
    
    processing_time = time.time() - start_time
    return results, processing_time

# Test with sample data
sample_data = [f"data_item_{i}" for i in range(20)]
results, duration = process_data_batch(sample_data)

print(f"Processed {len(sample_data)} items in {duration:.3f}s")
print(f"Average processing time: {duration/len(sample_data)*1000:.2f}ms per item")
print(f"Sample results:")
for result in results[:3]:
    print(f"  ID {result['id']}: {result['original']} -> {result['encoded']}")

# Calculate encoding efficiency
total_original = sum(len(item.encode()) for item in sample_data)
total_encoded = sum(result['size'] for result in results)
print(f"Encoding efficiency: {total_encoded/total_original:.2f}x size")

In [ ]:
import base58

# Example 5: Error handling and input validation
def safe_base58_operations():
    """Demonstrate error handling in Base58 operations"""
    
    test_cases = [
        ("Valid data", b"Hello, World!"),
        ("Empty data", b""),
        ("Unicode text", "Hello, 世界! 🌍".encode('utf-8')),
        ("Binary data", bytes(range(256))),
        ("Invalid decode", "InvalidBase58String0OIl"),  # Contains invalid chars
    ]
    
    results = []
    
    for description, data in test_cases:
        if isinstance(data, str):
            # This is a decode test of an INVALID string: it must be rejected with ValueError
            try:
                base58.b58decode(data)
            except ValueError as e:
                assert "Invalid character" in str(e), f"{description}: unexpected message {e!r}"
                results.append(f"✓ {description}: rejected as expected ({type(e).__name__}, {e})")
            else:
                raise AssertionError(f"{description}: expected ValueError for invalid Base58 input {data!r}")
        else:
            # This is an encode test
            encoded = base58.b58encode(data)
            decoded = base58.b58decode(encoded)
            
            # Verify round-trip
            assert decoded == data, f"{description}: round-trip failed"
            results.append(f"✓ {description}: Round-trip successful")
    
    return results

# Run error handling tests
test_results = safe_base58_operations()
print("Base58 error handling and validation:")
for result in test_results:
    print(f"  {result}")

print("\nError handling patterns demonstrated")

In [ ]:
import base58
import json
import time

# Example 6: Encode JSON payloads as Base58 text
class Base58Encoder:
    """Base58 encoder for JSON payloads that counts its calls"""
    
    def __init__(self):
        self.stats = {'encoded': 0, 'decoded': 0, 'errors': 0}
    
    def encode_json(self, data):
        """Encode JSON data to Base58"""
        try:
            json_str = json.dumps(data, separators=(',', ':'))
            json_bytes = json_str.encode('utf-8')
            encoded = base58.b58encode(json_bytes)
            self.stats['encoded'] += 1
            return encoded.decode()
        except Exception as e:
            self.stats['errors'] += 1
            raise e
    
    def decode_json(self, encoded_data):
        """Decode Base58 to JSON data"""
        try:
            decoded_bytes = base58.b58decode(encoded_data)
            json_str = decoded_bytes.decode('utf-8')
            data = json.loads(json_str)
            self.stats['decoded'] += 1
            return data
        except Exception as e:
            self.stats['errors'] += 1
            raise e
    
    def get_stats(self):
        return self.stats.copy()

# Encode three JSON objects
encoder = Base58Encoder()

# Test data representing different use cases
test_objects = [
    {'user_id': 12345, 'action': 'login', 'timestamp': int(time.time())},
    {'transaction_id': 'tx_789', 'amount': 100.50, 'currency': 'USD'},
    {'config': {'debug': True, 'max_connections': 100, 'timeout': 30}}
]

print("JSON objects encoded as Base58:")
encoded_objects = []

for i, obj in enumerate(test_objects):
    encoded = encoder.encode_json(obj)
    encoded_objects.append(encoded)
    print(f"  Object {i+1}: {len(encoded)} chars -> {encoded[:30]}...")

# Decode back and verify
print("\nDecoding verification:")
for i, encoded in enumerate(encoded_objects):
    decoded = encoder.decode_json(encoded)
    assert decoded == test_objects[i], (i, decoded)
    print(f"  Object {i+1}: round-trip successful")

# Show statistics
stats = encoder.get_stats()
print(f"\nEncoder statistics: {stats}")
assert stats == {'encoded': 3, 'decoded': 3, 'errors': 0}, stats